In [ ]:
import requests
from bs4 import BeautifulSoup
import time #for sleep/rate limiting

base_Url = "http://books.toscrape.com/catalogue/"

#scrape page by page.
current_url = "http://books.toscrape.com/catalogue/page-3.html"

all_books = [] #store all books in a list

#while loop for time scraping.

while current_url:
    try:
        response = requests.get(current_url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser") #store the content of the page in a variable called soup

        books = soup.select(".product_pod") #select all the elements with the class product_pod and store them in a variable called books
        for book in books:
            item = {} #store book in dictiornary
            try:
                title_tag = book.select_one("h3 a") #select the title of the book
                item['title'] = title_tag['title'] if title_tag else "N/A" #if the title tag is not found, set the title to N/A
        
                price_tag = book.select_one(".price_color") #select the price of the book
                item['price'] = price_tag.get_text(strip=True) if price_tag else "£0.00" #if the price tag is not found, set the price to £0.00
                stock_tag = book.select_one('.instock.availability') #select the stock availability of the book
                item['stock'] = stock_tag.get_text(strip=True) if stock_tag else "Out of stock" #if the stock tag is not found, set the stock to out of stock

                print(item) #print the book details
                all_books.append(item) #add book to the list

            except Exception as e:
                print(f"Skipped book due to error: {e}")
                continue #continuous scraping the next book

        next_button = soup.select_one("li.next a") #next page
        if next_button:
            next_page = next_button['href']
            if "/catalogue/" in next_page:
                current_url = "http://books.toscrape.com/" + next_page
            else:
                current_url = base_Url + next_page
                time.sleep(1) #rate limiting
        else:
            print("\nReached the final page.Finishing scraping...")
            current_url = None #break the loop if there is no next page
    except requests.exceptions.RequestException as err:
        print(f"Network Error (Status {response.status_code})")
        break #break the loop if there is a network error
    except Exception as err:
        print(f"An unexpected error occurred: {err}")
        break #break the loop if there is any other error
print("Scraping completed successfully.")
print(f"Total books scraped: {len(all_books)}")

In [ ]:
#DATA STORAGE in csv and txt.

import csv

# Save to CSV
with open("books.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=["title", "price", "stock"])
    writer.writeheader()
    writer.writerows(all_books)
print("Saved to books.csv")

# Save to TXT
with open("books.txt", "w", encoding="utf-8") as txtfile:
    for book in all_books:
        txtfile.write(f"Title: {book['title']} | Price: {book['price']} | Stock: {book['stock']}\n")
print("Saved to books.txt")

In [ ]:
#DATA STORAGE in Excel.(Requires 'openpyxl' library)

import pandas as pd

# Save to Excel
df = pd.DataFrame(all_books)
df.to_excel("books.xlsx", index=False)
print("Saved to books.xlsx")

In [11]:
# 5. Save to Disk
df.to_csv("scraped_books_page_3.csv", index=False)

print("Success! File saved as 'scraped_books_page_3.csv'")

Success! File saved as 'scraped_books_page_3.csv'
